# Notebook 02: OCR Comparison

**Goal:** Compare 4 OCR engines on the same page images and pick the best one for text extraction.

**Engines evaluated:**
| # | Engine | Type | Key Strengths |
|---|--------|------|---------------|
| 1 | **Tesseract** | Local, open source | Free, supports many languages |
| 2 | **EasyOCR** | Local, deep learning | Better accuracy, GPU support |
| 3 | **Google Cloud Vision** | Cloud API | Best layout analysis |
| 4 | **Azure AI Vision** | Cloud API | Excellent accuracy, Read API |

**Input:** Page images from `data/page_images/` (output of Notebook 01)  
**Output:** OCR results (bounding boxes + text) saved as JSON in `data/ocr_results/<engine>/`

In [ ]:
# Install dependencies (run once)
# !pip install pytesseract easyocr Pillow opencv-python-headless matplotlib numpy
# !sudo apt-get install -y tesseract-ocr tesseract-ocr-eng
# For Google Vision: pip install google-cloud-vision
# For Azure Vision: pip install azure-ai-vision-imageanalysis

In [ ]:
import json
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import (
    DATA_DIR, PAGE_IMAGES_DIR, OCR_RESULTS_DIR,
    load_json, save_json, load_image, save_image,
    draw_bboxes, display_images, display_comparison,
)

# Load page images from Notebook 01 output
metadata = load_json(PAGE_IMAGES_DIR / "metadata.json")
page_images = []
for page_info in metadata["pages"]:
    img = load_image(PAGE_IMAGES_DIR / page_info["filename"])
    page_images.append(img)

print(f"Loaded {len(page_images)} page(s) from data/page_images/")
for i, img in enumerate(page_images):
    print(f"  Page {i}: {img.size[0]}x{img.size[1]}")

In [ ]:
# Select which page to run OCR comparison on
PAGE_INDEX = 0  # Change this to test different pages
test_image = page_images[PAGE_INDEX]
print(f"Using page {PAGE_INDEX} for comparison: {test_image.size[0]}x{test_image.size[1]}")

## Helper: Standardized OCR Output Format

All engines normalize their output to a common schema:
```json
[{"id": 0, "text": "...", "bbox": [x0, y0, x1, y1], "confidence": 0.95, "level": "line"}]
```

In [ ]:
def save_ocr_results(engine_name: str, text_blocks: list[dict], image: Image.Image, page_idx: int = 0):
    """Save OCR results and visualization for an engine."""
    engine_dir = OCR_RESULTS_DIR / engine_name
    engine_dir.mkdir(parents=True, exist_ok=True)
    
    # Save JSON results
    save_json(text_blocks, engine_dir / f"page_{page_idx}.json")
    
    # Save visualization with bounding boxes
    viz_image = draw_bboxes(image, text_blocks, color="red", width=2, show_text=False)
    save_image(viz_image, engine_dir / f"page_{page_idx}_viz.png")
    
    print(f"  [{engine_name}] Saved {len(text_blocks)} text blocks + visualization")
    return viz_image


def print_ocr_summary(engine_name: str, text_blocks: list[dict], elapsed: float):
    """Print summary of OCR results."""
    total_chars = sum(len(b["text"]) for b in text_blocks)
    avg_conf = np.mean([b["confidence"] for b in text_blocks]) if text_blocks else 0
    print(f"\n{'='*60}")
    print(f"  {engine_name.upper()}")
    print(f"  Blocks: {len(text_blocks)} | Characters: {total_chars} | Avg confidence: {avg_conf:.2f}")
    print(f"  Time: {elapsed:.2f}s")
    print(f"{'='*60}")
    # Show first 5 blocks as sample
    for b in text_blocks[:5]:
        text_preview = b["text"][:60] + ("..." if len(b["text"]) > 60 else "")
        print(f"  [{b['id']:3d}] conf={b['confidence']:.2f} | {text_preview}")

---
## Option 1: Tesseract OCR

Open-source, local engine. Uses `pytesseract.image_to_data()` which returns word-level detections with bounding boxes and confidence. We group words into lines using Tesseract's built-in hierarchy (block → paragraph → line → word).

In [ ]:
import pytesseract
from pytesseract import Output


def run_tesseract(image: Image.Image, lang: str = "eng") -> list[dict]:
    """
    Run Tesseract OCR and return standardized text blocks (line-level).
    
    Groups word-level detections into lines using Tesseract's hierarchy.
    """
    data = pytesseract.image_to_data(image, lang=lang, output_type=Output.DICT)
    
    # Group words into lines using (block_num, par_num, line_num) as key
    lines = {}
    n = len(data["text"])
    for i in range(n):
        text = data["text"][i].strip()
        conf = int(data["conf"][i])
        if conf < 0 or not text:  # Skip empty/invalid entries
            continue
        
        line_key = (data["block_num"][i], data["par_num"][i], data["line_num"][i])
        
        if line_key not in lines:
            lines[line_key] = {
                "words": [],
                "x0": data["left"][i],
                "y0": data["top"][i],
                "x1": data["left"][i] + data["width"][i],
                "y1": data["top"][i] + data["height"][i],
                "confidences": [],
            }
        
        line = lines[line_key]
        line["words"].append(text)
        line["confidences"].append(conf)
        # Expand bounding box to include this word
        line["x0"] = min(line["x0"], data["left"][i])
        line["y0"] = min(line["y0"], data["top"][i])
        line["x1"] = max(line["x1"], data["left"][i] + data["width"][i])
        line["y1"] = max(line["y1"], data["top"][i] + data["height"][i])
    
    # Convert to standardized format
    text_blocks = []
    for idx, (key, line) in enumerate(sorted(lines.items())):
        text_blocks.append({
            "id": idx,
            "text": " ".join(line["words"]),
            "bbox": [line["x0"], line["y0"], line["x1"], line["y1"]],
            "confidence": round(np.mean(line["confidences"]) / 100.0, 3),
            "level": "line",
        })
    
    return text_blocks


# Run Tesseract
t0 = time.time()
tesseract_results = run_tesseract(test_image)
tesseract_time = time.time() - t0

print_ocr_summary("tesseract", tesseract_results, tesseract_time)
tesseract_viz = save_ocr_results("tesseract", tesseract_results, test_image, PAGE_INDEX)

---
## Option 2: EasyOCR

Deep learning-based local engine. Generally better accuracy than Tesseract on complex layouts. Returns bounding boxes as polygon coordinates.

In [ ]:
import easyocr


def run_easyocr(image: Image.Image, langs: list[str] = ["en"]) -> list[dict]:
    """
    Run EasyOCR and return standardized text blocks.
    
    EasyOCR returns results as: [(bbox_polygon, text, confidence), ...]
    where bbox_polygon is [[x0,y0],[x1,y1],[x2,y2],[x3,y3]] (4 corners).
    """
    reader = easyocr.Reader(langs, gpu=False)  # Set gpu=True if CUDA available
    img_array = np.array(image)
    results = reader.readtext(img_array)
    
    text_blocks = []
    for idx, (polygon, text, confidence) in enumerate(results):
        # Convert polygon to axis-aligned bbox [x0, y0, x1, y1]
        xs = [p[0] for p in polygon]
        ys = [p[1] for p in polygon]
        bbox = [int(min(xs)), int(min(ys)), int(max(xs)), int(max(ys))]
        
        text_blocks.append({
            "id": idx,
            "text": text,
            "bbox": bbox,
            "confidence": round(float(confidence), 3),
            "level": "line",
        })
    
    return text_blocks


# Run EasyOCR
t0 = time.time()
easyocr_results = run_easyocr(test_image)
easyocr_time = time.time() - t0

print_ocr_summary("easyocr", easyocr_results, easyocr_time)
easyocr_viz = save_ocr_results("easyocr", easyocr_results, test_image, PAGE_INDEX)

---
## Option 3: Google Cloud Vision

Google's cloud OCR API with excellent layout analysis. Requires a Google Cloud project with Vision API enabled and a service account key.

**Setup:** `pip install google-cloud-vision` and set `GOOGLE_APPLICATION_CREDENTIALS` env var.

In [ ]:
import io


def run_google_vision(image: Image.Image) -> list[dict]:
    """
    Run Google Cloud Vision OCR and return standardized text blocks.
    
    Requires: pip install google-cloud-vision
    Set env: GOOGLE_APPLICATION_CREDENTIALS=/path/to/service-account.json
    """
    from google.cloud import vision
    
    client = vision.ImageAnnotatorClient()
    
    # Convert PIL image to bytes
    buffer = io.BytesIO()
    image.save(buffer, format="PNG")
    content = buffer.getvalue()
    
    gv_image = vision.Image(content=content)
    response = client.document_text_detection(image=gv_image)
    
    if response.error.message:
        raise RuntimeError(f"Google Vision API error: {response.error.message}")
    
    text_blocks = []
    idx = 0
    
    # Parse full text annotation (structured by pages → blocks → paragraphs → words)
    for page in response.full_text_annotation.pages:
        for block in page.blocks:
            for paragraph in block.paragraphs:
                # Collect all words in this paragraph into a single line-level block
                words = []
                for word in paragraph.words:
                    word_text = "".join(s.text for s in word.symbols)
                    words.append(word_text)
                
                # Get paragraph bounding box
                vertices = paragraph.bounding_box.vertices
                xs = [v.x for v in vertices]
                ys = [v.y for v in vertices]
                bbox = [min(xs), min(ys), max(xs), max(ys)]
                
                confidence = paragraph.confidence if hasattr(paragraph, 'confidence') else 0.9
                
                text_blocks.append({
                    "id": idx,
                    "text": " ".join(words),
                    "bbox": bbox,
                    "confidence": round(float(confidence), 3),
                    "level": "paragraph",
                })
                idx += 1
    
    return text_blocks


# Run Google Vision (uncomment to use)
# Requires GOOGLE_APPLICATION_CREDENTIALS env var to be set
try:
    t0 = time.time()
    google_vision_results = run_google_vision(test_image)
    google_vision_time = time.time() - t0
    
    print_ocr_summary("google_vision", google_vision_results, google_vision_time)
    google_vision_viz = save_ocr_results("google_vision", google_vision_results, test_image, PAGE_INDEX)
except Exception as e:
    print(f"Google Vision skipped: {e}")
    print("Set GOOGLE_APPLICATION_CREDENTIALS and install google-cloud-vision to enable.")
    google_vision_results = None
    google_vision_viz = None

---
## Option 4: Azure AI Vision

Azure's Read API provides excellent OCR accuracy with built-in layout understanding. Requires an Azure Computer Vision resource.

**Setup:** `pip install azure-ai-vision-imageanalysis` and set `AZURE_VISION_ENDPOINT` + `AZURE_VISION_KEY` env vars.

In [ ]:
def run_azure_vision(image: Image.Image) -> list[dict]:
    """
    Run Azure AI Vision Read API and return standardized text blocks.
    
    Requires: pip install azure-ai-vision-imageanalysis
    Set env: AZURE_VISION_ENDPOINT, AZURE_VISION_KEY
    """
    from azure.ai.vision.imageanalysis import ImageAnalysisClient
    from azure.ai.vision.imageanalysis.models import VisualFeatures
    from azure.core.credentials import AzureKeyCredential
    
    endpoint = os.environ["AZURE_VISION_ENDPOINT"]
    key = os.environ["AZURE_VISION_KEY"]
    
    client = ImageAnalysisClient(endpoint=endpoint, credential=AzureKeyCredential(key))
    
    # Convert PIL image to bytes
    buffer = io.BytesIO()
    image.save(buffer, format="PNG")
    image_data = buffer.getvalue()
    
    result = client.analyze(
        image_data=image_data,
        visual_features=[VisualFeatures.READ],
    )
    
    text_blocks = []
    idx = 0
    
    if result.read and result.read.blocks:
        for block in result.read.blocks:
            for line in block.lines:
                # Azure returns bounding polygon as list of Points
                points = line.bounding_polygon
                xs = [p.x for p in points]
                ys = [p.y for p in points]
                bbox = [int(min(xs)), int(min(ys)), int(max(xs)), int(max(ys))]
                
                # Average word-level confidence
                confidences = [w.confidence for w in line.words if hasattr(w, 'confidence')]
                avg_conf = np.mean(confidences) if confidences else 0.9
                
                text_blocks.append({
                    "id": idx,
                    "text": line.text,
                    "bbox": bbox,
                    "confidence": round(float(avg_conf), 3),
                    "level": "line",
                })
                idx += 1
    
    return text_blocks


# Run Azure Vision (uncomment to use)
# Requires AZURE_VISION_ENDPOINT and AZURE_VISION_KEY env vars
try:
    t0 = time.time()
    azure_vision_results = run_azure_vision(test_image)
    azure_vision_time = time.time() - t0
    
    print_ocr_summary("azure_vision", azure_vision_results, azure_vision_time)
    azure_vision_viz = save_ocr_results("azure_vision", azure_vision_results, test_image, PAGE_INDEX)
except Exception as e:
    print(f"Azure Vision skipped: {e}")
    print("Set AZURE_VISION_ENDPOINT and AZURE_VISION_KEY and install azure-ai-vision-imageanalysis to enable.")
    azure_vision_results = None
    azure_vision_viz = None

---
## Side-by-Side Comparison

Visual comparison of bounding box detections from all available engines.

In [ ]:
# Collect all available results for comparison
all_results = {}
all_viz = {}
all_times = {}

if tesseract_results:
    all_results["Tesseract"] = tesseract_results
    all_viz["Tesseract"] = tesseract_viz
    all_times["Tesseract"] = tesseract_time

if easyocr_results:
    all_results["EasyOCR"] = easyocr_results
    all_viz["EasyOCR"] = easyocr_viz
    all_times["EasyOCR"] = easyocr_time

if google_vision_results:
    all_results["Google Vision"] = google_vision_results
    all_viz["Google Vision"] = google_vision_viz
    all_times["Google Vision"] = google_vision_time

if azure_vision_results:
    all_results["Azure Vision"] = azure_vision_results
    all_viz["Azure Vision"] = azure_vision_viz
    all_times["Azure Vision"] = azure_vision_time

# Display visualizations side by side
viz_images = list(all_viz.values())
viz_titles = [
    f"{name}\n{len(all_results[name])} blocks | {all_times[name]:.1f}s"
    for name in all_viz.keys()
]

if viz_images:
    display_images(viz_images, titles=viz_titles, cols=min(len(viz_images), 2), figsize=(16, 12))
else:
    print("No OCR results to display.")

In [ ]:
# Summary comparison table
print(f"{'Engine':<18} {'Blocks':>7} {'Characters':>11} {'Avg Conf':>10} {'Time (s)':>10}")
print("-" * 60)
for name, results in all_results.items():
    n_blocks = len(results)
    n_chars = sum(len(b["text"]) for b in results)
    avg_conf = np.mean([b["confidence"] for b in results]) if results else 0
    t = all_times[name]
    print(f"{name:<18} {n_blocks:>7} {n_chars:>11} {avg_conf:>10.3f} {t:>10.2f}")

## Select Best OCR Engine

After reviewing the visual comparison and summary above, set `BEST_ENGINE` to your preferred choice. This copies the results to `data/ocr_results/best/` for use by subsequent notebooks.

In [ ]:
# ── SELECT YOUR BEST ENGINE ────────────────────────────────────────────────────
# Options: "tesseract", "easyocr", "google_vision", "azure_vision"
BEST_ENGINE = "easyocr"
# ──────────────────────────────────────────────────────────────────────────────

import shutil

source_dir = OCR_RESULTS_DIR / BEST_ENGINE
best_dir = OCR_RESULTS_DIR / "best"

if best_dir.exists():
    shutil.rmtree(best_dir)
best_dir.mkdir(parents=True, exist_ok=True)

# Copy results from chosen engine to best/
for f in source_dir.iterdir():
    shutil.copy2(f, best_dir / f.name)

print(f"Copied {BEST_ENGINE} results to data/ocr_results/best/")
print(f"Files: {[f.name for f in best_dir.iterdir()]}")
print(f"\n✓ OCR selection complete. Next notebook: 03_translation_comparison.ipynb")